# ARC-AGI-3 Hybrid Graph Explorer Submission

In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.6-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatib

In [2]:

from __future__ import annotations

import gc
import hashlib
import json
import logging
import math
import os
import random
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from arc_agi import Arcade, OperationMode
from arcengine import GameAction, GameState

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception:
    torch = None
    nn = None
    F = None


SEED = 20260330
random.seed(SEED)
np.random.seed(SEED)

KAGGLE_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
DEFAULT_ENV_DIR = KAGGLE_ROOT / "environment_files"
ENVIRONMENTS_DIR = str(DEFAULT_ENV_DIR if DEFAULT_ENV_DIR.exists() else Path("environment_files"))

SAVE_RECORDINGS = False
INCLUDE_FRAME_DATA = False
MAX_RESETS_PER_LEVEL = 4
DEFAULT_BASELINE = 24
DEFAULT_LEVEL_BUDGET = 220
MAX_LEVEL_BUDGET = 420
MAX_WALL_CLOCK_SECONDS = float(os.environ.get("ARC_MAX_SECONDS", str(5.75 * 3600)))
N_GROUPS = 6
FORCE_TORCH_CPU = os.environ.get("ARC_FORCE_TORCH_CPU", "0") == "1"
USE_TORCH_MODEL = bool(torch is not None and (torch.cuda.is_available() or FORCE_TORCH_CPU))
DEVICE = torch.device("cuda" if (torch is not None and torch.cuda.is_available()) else "cpu") if USE_TORCH_MODEL and torch is not None else None

logger = logging.getLogger("arc_hybrid")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(handler)

INF = 10**9


def safe_int(x: Any, default: int = 0) -> int:
    try:
        return int(x)
    except Exception:
        return default


def normalize_frame(frame: np.ndarray, size: int = 64, fill_value: int = 0) -> np.ndarray:
    arr = np.asarray(frame, dtype=np.uint8)
    if arr.shape == (size, size):
        return arr
    out = np.full((size, size), fill_value, dtype=np.uint8)
    h = min(size, arr.shape[0])
    w = min(size, arr.shape[1])
    out[:h, :w] = arr[:h, :w]
    return out


def extract_last_frame(obs: Any) -> np.ndarray:
    if obs is None:
        return np.zeros((64, 64), dtype=np.uint8)
    frames = getattr(obs, "frame", None)
    if frames is None or len(frames) == 0:
        return np.zeros((64, 64), dtype=np.uint8)
    return normalize_frame(np.asarray(frames[-1], dtype=np.uint8))


def obs_state(obs: Any) -> GameState:
    state = getattr(obs, "state", GameState.NOT_PLAYED)
    if isinstance(state, GameState):
        return state
    try:
        return GameState(state)
    except Exception:
        return GameState.NOT_PLAYED


def obs_levels_completed(obs: Any) -> int:
    if hasattr(obs, "levels_completed"):
        return safe_int(getattr(obs, "levels_completed"), 0)
    if hasattr(obs, "score"):
        return safe_int(getattr(obs, "score"), 0)
    return 0


def obs_win_levels(obs: Any) -> int:
    return safe_int(getattr(obs, "win_levels", 0), 0)


def obs_available_actions(obs: Any) -> List[int]:
    actions = list(getattr(obs, "available_actions", []) or [])
    out: List[int] = []
    for a in actions:
        if isinstance(a, GameAction):
            out.append(a.value)
        else:
            try:
                out.append(int(a))
            except Exception:
                pass
    return sorted(set(out))


class FrameAnalyzer:
    OFFSETS4 = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self) -> None:
        self.frame_shape = (64, 64)
        self.status_bar_distance_threshold = 3
        self.status_bar_ratio_threshold = 5.0
        self.status_bar_twins_threshold = 3
        self.minimal_width = 2
        self.maximal_width = 32
        self.salient_colors = set(range(6, 16))
        self.non_salient_colors = set(range(0, 6))
        self.status_bar_color = 16

    def segment_frame(self, frame: np.ndarray) -> Tuple[np.ndarray, List[dict]]:
        h, w = frame.shape
        labels = np.full((h, w), -1, dtype=np.int32)
        components: List[dict] = []
        cid = -1

        for y in range(h):
            for x in range(w):
                if labels[y, x] != -1:
                    continue
                cid += 1
                color = int(frame[y, x])
                q = deque([(y, x)])
                labels[y, x] = cid

                min_x = max_x = x
                min_y = max_y = y
                area = 0

                while q:
                    cy, cx = q.popleft()
                    area += 1
                    min_x = min(min_x, cx)
                    max_x = max(max_x, cx)
                    min_y = min(min_y, cy)
                    max_y = max(max_y, cy)

                    for dy, dx in self.OFFSETS4:
                        ny, nx = cy + dy, cx + dx
                        if (
                            0 <= ny < h
                            and 0 <= nx < w
                            and labels[ny, nx] == -1
                            and int(frame[ny, nx]) == color
                        ):
                            labels[ny, nx] = cid
                            q.append((ny, nx))

                rect_area = (max_x - min_x + 1) * (max_y - min_y + 1)
                is_rect = area == rect_area
                components.append(
                    {
                        "bounding_box": (min_x, min_y, max_x, max_y),
                        "color": color,
                        "area": area,
                        "is_rectangle": is_rect,
                    }
                )

        for i, comp in enumerate(components):
            twins = []
            for j, other in enumerate(components):
                if i == j:
                    continue
                if (
                    other["area"] == comp["area"]
                    and other["is_rectangle"] == comp["is_rectangle"]
                    and other["color"] == comp["color"]
                ):
                    twins.append(j)
            comp["number_of_twins"] = len(twins)
            comp["twin_ids"] = twins

        return labels, components

    def check_segment_fully_on_edge(self, segment: dict, edges: Optional[List[str]] = None) -> List[str]:
        x1, y1, x2, y2 = segment["bounding_box"]
        edges = edges or ["any"]
        result: List[str] = []
        h, w = self.frame_shape

        if "left" in edges or "any" in edges:
            if max(x1, x2) < self.status_bar_distance_threshold:
                result.append("left")
        if "right" in edges or "any" in edges:
            if min(x1, x2) >= w - self.status_bar_distance_threshold:
                result.append("right")
        if "top" in edges or "any" in edges:
            if max(y1, y2) < self.status_bar_distance_threshold:
                result.append("top")
        if "bottom" in edges or "any" in edges:
            if min(y1, y2) >= h - self.status_bar_distance_threshold:
                result.append("bottom")
        return result

    def check_segment_ratio(self, segment: dict, direction: str = "any") -> bool:
        x1, y1, x2, y2 = segment["bounding_box"]
        width = x2 - x1 + 1
        height = y2 - y1 + 1
        ratio = width / max(height, 1)
        if ratio >= self.status_bar_ratio_threshold and direction in ("any", "horizontal"):
            return True
        if ratio <= 1.0 / self.status_bar_ratio_threshold and direction in ("any", "vertical"):
            return True
        return False

    def segment_twins_on_edge(self, segment: dict, components: List[dict], edges: Optional[List[str]] = None) -> List[int]:
        edges = edges or self.check_segment_fully_on_edge(segment, ["any"])
        if not edges:
            return []
        out = []
        for twin_id in segment["twin_ids"]:
            twin = components[twin_id]
            if self.check_segment_fully_on_edge(twin, edges):
                out.append(twin_id)
        return out

    def identify_status_mask(self, labels: np.ndarray, components: List[dict]) -> np.ndarray:
        checked: set[int] = set()
        status_groups: List[List[int]] = []

        for i, comp in enumerate(components):
            if i in checked:
                continue
            checked.add(i)

            on_edges = self.check_segment_fully_on_edge(comp, ["any"])
            if not on_edges:
                continue

            directions = []
            if "left" in on_edges or "right" in on_edges:
                directions.append("vertical")
            if "top" in on_edges or "bottom" in on_edges:
                directions.append("horizontal")
            direction = "any" if len(directions) == 2 else directions[0]

            status_ids = [i]
            is_long = self.check_segment_ratio(comp, direction)

            if not is_long:
                twin_ids = self.segment_twins_on_edge(comp, components, on_edges)
                for tid in twin_ids:
                    checked.add(tid)
                if len(twin_ids) + 1 < self.status_bar_twins_threshold:
                    continue
                status_ids.extend(twin_ids)

            status_groups.append(status_ids)

        mask = np.zeros(labels.shape, dtype=bool)
        for group in status_groups:
            for cid in group:
                mask[labels == cid] = True
        return mask

    def hash_frame(self, frame: np.ndarray) -> str:
        arr = np.asarray(frame, dtype=np.uint8, order="C")
        flat = arr.ravel()
        if flat.size & 1:
            flat = np.concatenate([flat, np.zeros(1, dtype=np.uint8)])
        packed = (flat[0::2] << 4) | (flat[1::2] & 0x0F)
        payload = packed.tobytes()
        shape_tag = repr(arr.shape).encode()
        return hashlib.blake2b(payload, digest_size=16, person=shape_tag).hexdigest()

    def closest_pixel(self, pixels_yx: np.ndarray, target_x: float, target_y: float) -> Tuple[int, int]:
        if len(pixels_yx) == 0:
            return 0, 0
        dy = pixels_yx[:, 0] - target_y
        dx = pixels_yx[:, 1] - target_x
        idx = int(np.argmin(dx * dx + dy * dy))
        y, x = pixels_yx[idx]
        return int(x), int(y)

    def prepare_frame(self, raw_frame: np.ndarray, status_mask: Optional[np.ndarray] = None) -> dict:
        raw_frame = normalize_frame(raw_frame)
        if status_mask is None:
            labels_raw, comps_raw = self.segment_frame(raw_frame)
            status_mask = self.identify_status_mask(labels_raw, comps_raw)

        seg_frame = raw_frame.copy()
        seg_frame[status_mask] = self.status_bar_color
        canonical = raw_frame.copy()
        canonical[status_mask] = 0

        labels, components = self.segment_frame(seg_frame)
        frame_hash = self.hash_frame(canonical)

        return {
            "raw": raw_frame,
            "canonical": canonical,
            "seg_frame": seg_frame,
            "labels": labels,
            "components": components,
            "status_mask": status_mask,
            "hash": frame_hash,
        }

    def component_base_group(self, comp: dict) -> int:
        x1, y1, x2, y2 = comp["bounding_box"]
        width = x2 - x1 + 1
        height = y2 - y1 + 1
        is_status = comp["color"] == self.status_bar_color
        salient = comp["color"] in self.salient_colors
        medium = (
            self.minimal_width <= width <= self.maximal_width
            and self.minimal_width <= height <= self.maximal_width
        )
        if is_status:
            return 5
        if salient and medium:
            return 1
        if salient or medium:
            return 2
        return 3

    def build_click_candidates(self, prepared: dict, include_grid8: bool = True) -> List["Candidate"]:
        raw = prepared["raw"]
        labels = prepared["labels"]
        components = prepared["components"]

        candidates: List[Candidate] = []
        seen_xy: set[Tuple[int, int, str]] = set()
        seen_coords: set[Tuple[int, int]] = set()

        def add_click(
            x: int,
            y: int,
            group: int,
            point_type: str,
            comp: Optional[dict] = None,
            extra: Optional[Dict[str, Any]] = None,
        ) -> None:
            x = int(max(0, min(63, x)))
            y = int(max(0, min(63, y)))
            key = (x, y, point_type)
            if key in seen_xy:
                return
            seen_xy.add(key)
            seen_coords.add((x, y))
            seg_color = int(comp["color"]) if comp is not None else None
            bbox = tuple(comp["bounding_box"]) if comp is not None else None
            area = int(comp["area"]) if comp is not None else None
            x1, y1, x2, y2 = bbox if bbox is not None else (x, y, x, y)
            width = x2 - x1 + 1
            height = y2 - y1 + 1
            salient = bool(seg_color is not None and seg_color in self.salient_colors)
            on_edge = bool(
                x <= 0 or y <= 0 or x >= 63 or y >= 63 or (
                    comp is not None and bool(self.check_segment_fully_on_edge(comp, ["any"]))
                )
            )
            payload = extra or {}
            candidates.append(
                Candidate(
                    kind="click",
                    group=int(max(1, min(N_GROUPS - 1, group))),
                    action_id=6,
                    x=x,
                    y=y,
                    point_type=point_type,
                    cell_color=int(raw[y, x]),
                    segment_color=seg_color,
                    area=area,
                    width=width,
                    height=height,
                    salient=salient,
                    on_edge=on_edge,
                    status_like=bool(seg_color == self.status_bar_color),
                    meta=payload,
                )
            )

        for cid, comp in enumerate(components):
            pixels = np.argwhere(labels == cid)
            if len(pixels) == 0:
                continue
            x1, y1, x2, y2 = comp["bounding_box"]
            width = x2 - x1 + 1
            height = y2 - y1 + 1
            group = self.component_base_group(comp)

            cx, cy = self.closest_pixel(pixels, (x1 + x2) / 2.0, (y1 + y2) / 2.0)
            add_click(cx, cy, group, "segment_center", comp, {"component_id": cid})

            area = int(comp["area"])
            elongated = max(width, height) >= 6 and max(width, height) >= 2 * max(1, min(width, height))
            large = area >= 12 or max(width, height) >= 5

            extra_targets: List[Tuple[float, float, str]] = []
            if large:
                extra_targets.extend([
                    (x1, y1, "segment_tl"),
                    (x2, y2, "segment_br"),
                ])
            if elongated:
                if width >= height:
                    extra_targets.extend([
                        (x1, (y1 + y2) / 2.0, "segment_left"),
                        (x2, (y1 + y2) / 2.0, "segment_right"),
                    ])
                else:
                    extra_targets.extend([
                        (((x1 + x2) / 2.0), y1, "segment_top"),
                        (((x1 + x2) / 2.0), y2, "segment_bottom"),
                    ])

            added_pt_types = {"segment_center"}
            for tx, ty, pt_type in extra_targets:
                if pt_type in added_pt_types:
                    continue
                px, py = self.closest_pixel(pixels, tx, ty)
                add_click(px, py, min(group + 1, N_GROUPS - 1), pt_type, comp, {"component_id": cid})
                added_pt_types.add(pt_type)

            if group <= 3:
                outside_points = [
                    ((x1 + x2) // 2, max(0, y1 - 1), "adj_top"),
                    ((x1 + x2) // 2, min(63, y2 + 1), "adj_bottom"),
                    (max(0, x1 - 1), (y1 + y2) // 2, "adj_left"),
                    (min(63, x2 + 1), (y1 + y2) // 2, "adj_right"),
                ]
                for px, py, pt_type in outside_points:
                    add_click(px, py, min(group + 2, N_GROUPS - 1), pt_type, comp, {"component_id": cid})

        special_points = [
            (0, 0, "corner"),
            (63, 0, "corner"),
            (0, 63, "corner"),
            (63, 63, "corner"),
            (31, 31, "center"),
            (31, 0, "edge_mid"),
            (31, 63, "edge_mid"),
            (0, 31, "edge_mid"),
            (63, 31, "edge_mid"),
            (15, 15, "quarter"),
            (47, 15, "quarter"),
            (15, 47, "quarter"),
            (47, 47, "quarter"),
        ]
        for x, y, point_type in special_points:
            add_click(x, y, 4, point_type, None, {})

        grid4 = [8, 24, 40, 56]
        for gy in grid4:
            for gx in grid4:
                add_click(gx, gy, 4, "grid4", None, {})

        if include_grid8 and len(candidates) < 90:
            grid8 = [4, 12, 20, 28, 36, 44, 52, 60]
            for gy in grid8:
                for gx in grid8:
                    if (gx, gy) in seen_coords:
                        continue
                    add_click(gx, gy, 5, "grid8", None, {})

        candidates.sort(key=lambda c: (c.group, c.kind, c.action_id if c.action_id is not None else 99, c.y or 0, c.x or 0, c.point_type))
        for idx, cand in enumerate(candidates):
            cand.idx = idx
        return candidates


@dataclass
class Candidate:
    kind: str
    group: int
    action_id: Optional[int] = None
    x: Optional[int] = None
    y: Optional[int] = None
    point_type: str = ""
    cell_color: int = 0
    segment_color: Optional[int] = None
    area: Optional[int] = None
    width: int = 1
    height: int = 1
    salient: bool = False
    on_edge: bool = False
    status_like: bool = False
    meta: Dict[str, Any] = field(default_factory=dict)
    idx: int = -1

    def predictor_index(self) -> Optional[int]:
        if self.kind == "simple" and self.action_id is not None:
            return int(self.action_id) - 1
        if self.kind == "click" and self.x is not None and self.y is not None:
            return 7 + int(self.y) * 64 + int(self.x)
        return None

    def action_enum(self) -> GameAction:
        if self.kind == "simple" and self.action_id is not None:
            return GameAction.from_id(int(self.action_id))
        return GameAction.ACTION6

    def action_data(self) -> Optional[dict]:
        if self.kind == "click" and self.x is not None and self.y is not None:
            return {"x": int(self.x), "y": int(self.y)}
        return None


@dataclass
class StatePayload:
    frame_hash: str
    canonical_frame: np.ndarray
    raw_frame: np.ndarray
    candidates: List[Candidate]
    group_sets: List[set]


@dataclass
class NodeInfo:
    name: str
    total_candidates: int
    n_groups: int
    group2remaining_candidate_ids: List[set]
    edge_group: List[int] = field(default_factory=list)
    edge_result: List[int] = field(default_factory=list)   # 0 untested, 1 success, -1 fail
    edge_target: List[Optional[str]] = field(default_factory=list)
    edge_distance: List[int] = field(default_factory=list)
    distance: int = INF
    closed: bool = False

    def __post_init__(self) -> None:
        if not self.edge_group:
            self.edge_group = [0] * self.total_candidates
            for group_id, remaining in enumerate(self.group2remaining_candidate_ids):
                for idx in remaining:
                    if 0 <= idx < self.total_candidates:
                        self.edge_group[idx] = group_id
        if not self.edge_result:
            self.edge_result = [0] * self.total_candidates
        if not self.edge_target:
            self.edge_target = [None] * self.total_candidates
        if not self.edge_distance:
            self.edge_distance = [INF] * self.total_candidates

    def has_open_group(self, active_group: int) -> bool:
        for gid in range(min(active_group, self.n_groups - 1) + 1):
            if self.group2remaining_candidate_ids[gid]:
                return True
        return False

    def record_test(self, edge_idx: int, success: bool, target_node: Optional[str]) -> None:
        if edge_idx < 0 or edge_idx >= self.total_candidates:
            return
        grp = self.edge_group[edge_idx]
        self.group2remaining_candidate_ids[grp].discard(edge_idx)
        if success:
            self.edge_result[edge_idx] = 1
            self.edge_target[edge_idx] = target_node
            self.edge_distance[edge_idx] = INF
        else:
            self.edge_result[edge_idx] = -1
            self.edge_target[edge_idx] = None
            self.edge_distance[edge_idx] = INF


class GraphExplorer:
    def __init__(self, n_groups: int = N_GROUPS) -> None:
        self.n_groups = n_groups
        self.reset()

    def reset(self) -> None:
        self.nodes: Dict[str, NodeInfo] = {}
        self.graph: Dict[str, set] = defaultdict(set)      # source -> {(edge_idx, target)}
        self.graph_rev: Dict[str, set] = defaultdict(set)  # target -> {(edge_idx, source)}
        self.frontier: set[str] = set()
        self.dist: Dict[str, int] = {}
        self.active_group: int = 0

    def _clone_groups(self, group_sets: Sequence[set], n_candidates: int) -> List[set]:
        groups = [set() for _ in range(self.n_groups)]
        for gid, group in enumerate(group_sets[: self.n_groups]):
            groups[gid] = {int(i) for i in group if 0 <= int(i) < n_candidates}
        assigned = set().union(*groups) if groups else set()
        if len(assigned) < n_candidates:
            missing = set(range(n_candidates)) - assigned
            groups[min(self.n_groups - 1, 0)] |= missing
        return groups

    def add_node(self, name: str, n_candidates: int, group_sets: Sequence[set]) -> None:
        if name in self.nodes:
            return
        groups = self._clone_groups(group_sets, n_candidates)
        node = NodeInfo(name=name, total_candidates=n_candidates, n_groups=self.n_groups, group2remaining_candidate_ids=groups)
        self.nodes[name] = node
        self.graph[name] = set()
        self.graph_rev[name] = set()
        self.refresh()

    def refresh(self) -> None:
        self.frontier = set()
        for name, node in self.nodes.items():
            node.closed = not node.has_open_group(self.active_group)
            if not node.closed:
                self.frontier.add(name)
        self.rebuild_distances()

    def rebuild_distances(self) -> None:
        self.dist = {name: INF for name in self.nodes}
        for node in self.nodes.values():
            node.distance = INF
            for idx in range(node.total_candidates):
                if node.edge_result[idx] == 1:
                    node.edge_distance[idx] = INF

        dq = deque()
        for name in self.frontier:
            self.dist[name] = 0
            self.nodes[name].distance = 0
            dq.append(name)

        while dq:
            v = dq.popleft()
            vdist = self.dist[v]
            for edge_idx, src in self.graph_rev.get(v, ()):
                src_node = self.nodes[src]
                if src_node.edge_result[edge_idx] != 1:
                    continue
                if src_node.edge_group[edge_idx] > self.active_group:
                    continue
                candidate_dist = vdist + 1
                if candidate_dist < src_node.edge_distance[edge_idx]:
                    src_node.edge_distance[edge_idx] = candidate_dist
                if candidate_dist < self.dist[src]:
                    self.dist[src] = candidate_dist
                    src_node.distance = candidate_dist
                    dq.append(src)

    def ensure_reachable(self, name: str) -> None:
        while name in self.nodes and self.nodes[name].distance >= INF and self.active_group < self.n_groups - 1:
            self.active_group += 1
            self.refresh()

    def record_test(
        self,
        node_name: str,
        edge_idx: int,
        success: bool,
        target_node: Optional[str] = None,
        target_num_candidates: Optional[int] = None,
        target_group_sets: Optional[Sequence[set]] = None,
    ) -> None:
        if node_name not in self.nodes:
            return
        node = self.nodes[node_name]
        if node.edge_result[edge_idx] == 1 and node.edge_target[edge_idx] == target_node:
            return
        if node.edge_result[edge_idx] == -1 and not success:
            return

        # remove old edge if overwriting a success
        old_target = node.edge_target[edge_idx]
        if old_target is not None:
            self.graph[node_name].discard((edge_idx, old_target))
            self.graph_rev[old_target].discard((edge_idx, node_name))

        node.record_test(edge_idx=edge_idx, success=success, target_node=target_node)

        if success and target_node is not None:
            if target_node not in self.nodes:
                if target_num_candidates is None or target_group_sets is None:
                    return
                self.add_node(target_node, target_num_candidates, target_group_sets)
            self.graph[node_name].add((edge_idx, target_node))
            self.graph_rev[target_node].add((edge_idx, node_name))

        self.refresh()


class FeatureBandit:
    def __init__(self) -> None:
        self.change_stats: Dict[str, List[float]] = defaultdict(lambda: [0.0, 0.0])
        self.progress_stats: Dict[str, List[float]] = defaultdict(lambda: [0.0, 0.0])
        self.danger_stats: Dict[str, List[float]] = defaultdict(lambda: [0.0, 0.0])
        self.total_updates = 0.0

    def _bucket(self, value: Optional[int], bins: Sequence[int], labels: Sequence[str]) -> str:
        if value is None:
            return "none"
        for threshold, label in zip(bins, labels):
            if value <= threshold:
                return label
        return labels[-1]

    def feature_tokens(self, cand: Candidate) -> List[str]:
        tokens: List[str] = [f"kind:{cand.kind}", f"group:{cand.group}"]
        if cand.kind == "simple":
            tokens.append(f"simple:{cand.action_id}")
            tokens.append(f"exact:simple:{cand.action_id}")
            return tokens

        tokens.append(f"ptype:{cand.point_type}")
        tokens.append(f"cell:{cand.cell_color}")
        tokens.append(f"xbin:{(cand.x or 0) // 16}")
        tokens.append(f"ybin:{(cand.y or 0) // 16}")
        tokens.append(f"on_edge:{int(cand.on_edge)}")
        tokens.append(f"status_like:{int(cand.status_like)}")
        tokens.append(f"salient:{int(cand.salient)}")
        if cand.segment_color is not None:
            tokens.append(f"segcolor:{cand.segment_color}")
        area_bucket = self._bucket(cand.area, [2, 6, 20, 80], ["a1", "a2", "a3", "a4"])
        width_bucket = self._bucket(cand.width, [1, 3, 8, 64], ["w1", "w2", "w3", "w4"])
        height_bucket = self._bucket(cand.height, [1, 3, 8, 64], ["h1", "h2", "h3", "h4"])
        tokens.extend([f"area:{area_bucket}", f"width:{width_bucket}", f"height:{height_bucket}"])
        tokens.append(f"exact:click:{cand.point_type}:{cand.x}:{cand.y}")
        return tokens

    def _posterior_mean(self, stats: List[float], alpha0: float = 1.0, beta0: float = 1.0) -> float:
        succ, fail = stats
        return (succ + alpha0) / (succ + fail + alpha0 + beta0)

    def _ucb(self, stats: List[float]) -> float:
        n = stats[0] + stats[1]
        return math.sqrt(math.log(self.total_updates + 2.0) / (n + 1.0))

    def score(self, cand: Candidate) -> float:
        tokens = self.feature_tokens(cand)
        if not tokens:
            return 0.5
        change = []
        progress = []
        danger = []
        bonus = []
        for t in tokens:
            change.append(self._posterior_mean(self.change_stats[t], 1.0, 1.0))
            progress.append(self._posterior_mean(self.progress_stats[t], 0.5, 4.0))
            danger.append(self._posterior_mean(self.danger_stats[t], 0.5, 2.0))
            bonus.append(self._ucb(self.change_stats[t]))
        return (
            0.58 * float(np.mean(change))
            + 0.92 * float(np.mean(progress))
            - 0.62 * float(np.mean(danger))
            + 0.12 * float(np.mean(bonus))
        )

    def update(self, cand: Candidate, changed: bool, progress: bool, danger: bool) -> None:
        tokens = self.feature_tokens(cand)
        for t in tokens:
            if changed or progress:
                self.change_stats[t][0] += 1.0
            else:
                self.change_stats[t][1] += 1.0

            if progress:
                self.progress_stats[t][0] += 1.0
            else:
                self.progress_stats[t][1] += 0.05

            if danger:
                self.danger_stats[t][0] += 1.0
            else:
                self.danger_stats[t][1] += 0.05

        self.total_updates += 1.0


if torch is not None:
    class TinyActionModel(nn.Module):
        def __init__(self, in_channels: int = 16, num_simple_actions: int = 7) -> None:
            super().__init__()
            self.num_simple_actions = num_simple_actions
            self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
            self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
            self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

            self.action_pool = nn.AdaptiveAvgPool2d(1)
            self.action_fc1 = nn.Linear(64, 64)
            self.action_fc2 = nn.Linear(64, num_simple_actions)

            self.coord_conv1 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
            self.coord_conv2 = nn.Conv2d(32, 1, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = F.relu(self.conv1(x))
            x = F.relu(self.conv2(x))
            feat = F.relu(self.conv3(x))

            action_feat = self.action_pool(feat).flatten(1)
            action_feat = F.relu(self.action_fc1(action_feat))
            action_logits = self.action_fc2(action_feat)

            coord_feat = F.relu(self.coord_conv1(feat))
            coord_logits = self.coord_conv2(coord_feat).flatten(1)

            return torch.cat([action_logits, coord_logits], dim=1)


class OnlinePredictor:
    def __init__(self, enabled: bool) -> None:
        self.enabled = bool(enabled and torch is not None and nn is not None and F is not None and DEVICE is not None)
        self.device = DEVICE
        self.buffer: deque = deque(maxlen=4096)
        self.recent_keys: set[str] = set()
        self.batch_size = 48
        self.train_every = 6
        self.updates = 0
        if self.enabled:
            self.model = TinyActionModel(in_channels=16, num_simple_actions=7).to(self.device)
            self.optimizer = torch.optim.Adam(self.model.parameters(), lr=2e-4)
        else:
            self.model = None
            self.optimizer = None

    def frame_to_tensor(self, canonical_frame: np.ndarray) -> Optional["torch.Tensor"]:
        if not self.enabled:
            return None
        frame = np.asarray(canonical_frame, dtype=np.int64)
        frame = np.clip(frame, 0, 15)
        tensor = torch.zeros((16, 64, 64), dtype=torch.float32)
        index = torch.from_numpy(frame).unsqueeze(0)
        tensor.scatter_(0, index, 1.0)
        return tensor.to(self.device)

    def _exp_key(self, frame_hash: str, action_idx: int) -> str:
        return f"{frame_hash}:{action_idx}"

    def record(self, frame_hash: str, canonical_frame: np.ndarray, cand: Candidate, changed: bool, progress: bool, danger: bool) -> None:
        if not self.enabled:
            return
        action_idx = cand.predictor_index()
        if action_idx is None:
            return
        key = self._exp_key(frame_hash, action_idx)
        if key in self.recent_keys:
            return
        reward = 1.0 if (changed or progress) and not danger else 0.0
        tensor = self.frame_to_tensor(canonical_frame)
        if tensor is None:
            return
        self.buffer.append((tensor.cpu(), int(action_idx), float(reward)))
        self.recent_keys.add(key)
        if len(self.recent_keys) > 12000:
            self.recent_keys = set(list(self.recent_keys)[-6000:])
        self.updates += 1
        if self.updates % self.train_every == 0:
            self.train_step()

    def train_step(self) -> None:
        if not self.enabled or self.model is None or self.optimizer is None:
            return
        if len(self.buffer) < self.batch_size:
            return

        idxs = np.random.choice(len(self.buffer), self.batch_size, replace=False)
        states = torch.stack([self.buffer[i][0] for i in idxs]).to(self.device)
        action_indices = torch.tensor([self.buffer[i][1] for i in idxs], dtype=torch.long, device=self.device)
        rewards = torch.tensor([self.buffer[i][2] for i in idxs], dtype=torch.float32, device=self.device)

        self.optimizer.zero_grad(set_to_none=True)
        logits = self.model(states)
        selected = logits.gather(1, action_indices.unsqueeze(1)).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(selected, rewards)
        loss.backward()
        self.optimizer.step()

    def score_candidates(self, canonical_frame: np.ndarray, candidates: Sequence[Candidate]) -> Dict[int, float]:
        if not self.enabled or self.model is None:
            return {}
        with torch.no_grad():
            tensor = self.frame_to_tensor(canonical_frame)
            if tensor is None:
                return {}
            logits = self.model(tensor.unsqueeze(0)).squeeze(0)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
        out: Dict[int, float] = {}
        for cand in candidates:
            idx = cand.predictor_index()
            if idx is not None and 0 <= idx < len(probs):
                out[cand.idx] = float(probs[idx])
        return out


class HybridGameAgent:
    def __init__(self, env_info: Any, seed: int = SEED) -> None:
        self.env_info = env_info
        self.game_id = env_info.game_id
        self.seed = int(seed + abs(hash(self.game_id)) % 1_000_003)
        self.rng = random.Random(self.seed)

        self.analyzer = FrameAnalyzer()
        self.bandit = FeatureBandit()
        self.predictor = OnlinePredictor(USE_TORCH_MODEL)

        self.graph = GraphExplorer(n_groups=N_GROUPS)
        self.state_payloads: Dict[str, StatePayload] = {}
        self.status_mask: Optional[np.ndarray] = None
        self.current_level_completed = -1
        self.level_action_count = 0
        self.level_reset_count = 0
        self.local_total_actions = 0
        self.local_total_resets = 0
        self.debug_last_hash: Optional[str] = None
        self.level_start_hash: Optional[str] = None

    def _baseline_for_current_level(self) -> int:
        baselines = getattr(self.env_info, "baseline_actions", None)
        idx = max(0, self.current_level_completed)
        if baselines and idx < len(baselines):
            return max(1, safe_int(baselines[idx], DEFAULT_BASELINE))
        return DEFAULT_BASELINE

    def _level_budget(self, available_actions: Sequence[int]) -> int:
        baseline = self._baseline_for_current_level()
        budget = int(max(DEFAULT_LEVEL_BUDGET, baseline * 8))
        if 6 in available_actions:
            budget += 40
        if 7 in available_actions:
            budget += 20
        return int(min(MAX_LEVEL_BUDGET, budget))

    def _new_level(self, obs: Any) -> None:
        raw = extract_last_frame(obs)
        labels, comps = self.analyzer.segment_frame(raw)
        self.status_mask = self.analyzer.identify_status_mask(labels, comps)
        prepared = self.analyzer.prepare_frame(raw, self.status_mask)
        self.level_start_hash = prepared["hash"]
        self.graph = GraphExplorer(n_groups=N_GROUPS)
        self.state_payloads = {}
        self.current_level_completed = obs_levels_completed(obs)
        self.level_action_count = 0
        self.level_reset_count = 0
        self.debug_last_hash = None

    def _simple_action_group(self, action_id: int) -> int:
        probe = Candidate(kind="simple", group=0, action_id=int(action_id), point_type=f"simple_{action_id}")
        token = f"simple:{int(action_id)}"
        stats = self.bandit.change_stats[token]
        total = stats[0] + stats[1]
        score = self.bandit.score(probe)
        if total >= 12 and score < 0.34:
            return 2
        if total >= 6 and score < 0.44:
            return 1
        return 0

    def _build_candidates(self, prepared: dict, available_actions: Sequence[int]) -> List[Candidate]:
        candidates: List[Candidate] = []
        for action_id in sorted(set(int(a) for a in available_actions)):
            if action_id == 0:
                continue
            if action_id == 6:
                continue
            if 1 <= action_id <= 7:
                simple_group = self._simple_action_group(int(action_id))
                candidates.append(
                    Candidate(
                        kind="simple",
                        group=simple_group,
                        action_id=int(action_id),
                        point_type=f"simple_{action_id}",
                        cell_color=0,
                        area=1,
                        width=1,
                        height=1,
                    )
                )
        if 6 in available_actions:
            clicks = self.analyzer.build_click_candidates(prepared, include_grid8=True)
            candidates.extend(clicks)

        candidates.sort(key=lambda c: (c.group, c.kind, c.action_id if c.action_id is not None else 99, c.y or 0, c.x or 0, c.point_type))
        for idx, cand in enumerate(candidates):
            cand.idx = idx
        return candidates

    def _group_sets(self, candidates: Sequence[Candidate]) -> List[set]:
        groups = [set() for _ in range(N_GROUPS)]
        for cand in candidates:
            groups[min(max(cand.group, 0), N_GROUPS - 1)].add(cand.idx)
        return groups

    def _get_state_payload(self, obs: Any) -> StatePayload:
        prepared = self.analyzer.prepare_frame(extract_last_frame(obs), self.status_mask)
        frame_hash = prepared["hash"]
        payload = self.state_payloads.get(frame_hash)
        if payload is not None:
            return payload

        candidates = self._build_candidates(prepared, obs_available_actions(obs))
        groups = self._group_sets(candidates)
        payload = StatePayload(
            frame_hash=frame_hash,
            canonical_frame=prepared["canonical"],
            raw_frame=prepared["raw"],
            candidates=candidates,
            group_sets=groups,
        )
        self.state_payloads[frame_hash] = payload
        self.graph.add_node(frame_hash, len(candidates), groups)
        return payload

    def _candidate_score(self, cand: Candidate, predictor_scores: Dict[int, float], follow_mode: bool = False) -> float:
        score = self.bandit.score(cand)
        if cand.idx in predictor_scores:
            score = 0.68 * score + 0.32 * predictor_scores[cand.idx]
        if cand.kind == "simple":
            score += 0.02
        if cand.kind == "click" and cand.point_type.startswith("segment_"):
            score += 0.01
        if follow_mode:
            score += 0.02
        score += 0.002 * self.rng.random()
        return score

    def choose_candidate(self, payload: StatePayload) -> Optional[Candidate]:
        node = self.graph.nodes[payload.frame_hash]
        self.graph.ensure_reachable(payload.frame_hash)
        node = self.graph.nodes[payload.frame_hash]

        predictor_scores = self.predictor.score_candidates(payload.canonical_frame, payload.candidates)

        if node.has_open_group(self.graph.active_group):
            option_ids: List[int] = []
            for gid in range(self.graph.active_group + 1):
                option_ids.extend(sorted(node.group2remaining_candidate_ids[gid]))
            if not option_ids:
                return None
            scored = []
            for idx in option_ids:
                cand = payload.candidates[idx]
                scored.append((self._candidate_score(cand, predictor_scores, follow_mode=False), idx))
            scored.sort(reverse=True)
            top_k = min(5, len(scored))
            chosen_idx = scored[self.rng.randrange(top_k)][1] if top_k > 1 else scored[0][1]
            return payload.candidates[chosen_idx]

        viable = []
        best_dist = min(
            [
                node.edge_distance[idx]
                for idx in range(node.total_candidates)
                if node.edge_result[idx] == 1 and node.edge_group[idx] <= self.graph.active_group and node.edge_distance[idx] < INF
            ] or [INF]
        )
        if best_dist >= INF:
            return None
        for idx in range(node.total_candidates):
            if node.edge_result[idx] == 1 and node.edge_group[idx] <= self.graph.active_group and node.edge_distance[idx] == best_dist:
                viable.append(idx)
        if not viable:
            return None

        scored = []
        for idx in viable:
            cand = payload.candidates[idx]
            scored.append((self._candidate_score(cand, predictor_scores, follow_mode=True), idx))
        scored.sort(reverse=True)
        chosen_idx = scored[0][1]
        return payload.candidates[chosen_idx]

    def should_retry(self, obs: Any) -> bool:
        if self.level_reset_count >= MAX_RESETS_PER_LEVEL:
            return False
        budget = self._level_budget(obs_available_actions(obs))
        if self.level_action_count >= budget:
            return False
        if not self.graph.nodes:
            return False
        if self.level_start_hash is None or self.level_start_hash not in self.graph.nodes:
            return bool(self.graph.frontier)
        self.graph.ensure_reachable(self.level_start_hash)
        start_node = self.graph.nodes[self.level_start_hash]
        return start_node.has_open_group(self.graph.active_group) or start_node.distance < INF

    def should_stop_level(self, obs: Any) -> bool:
        budget = self._level_budget(obs_available_actions(obs))
        if self.level_action_count >= budget:
            return True
        payload = self._get_state_payload(obs)
        self.graph.ensure_reachable(payload.frame_hash)
        if not self.graph.frontier:
            return True
        node = self.graph.nodes[payload.frame_hash]
        if not node.has_open_group(self.graph.active_group) and node.distance >= INF and self.graph.active_group >= N_GROUPS - 1:
            return True
        return False

    def apply_transition(
        self,
        prev_payload: StatePayload,
        cand: Candidate,
        next_obs: Optional[Any],
    ) -> dict:
        result = {
            "changed": False,
            "progress": False,
            "danger": False,
            "next_payload": None,
        }

        if next_obs is None:
            self.bandit.update(cand, changed=False, progress=False, danger=True)
            self.predictor.record(prev_payload.frame_hash, prev_payload.canonical_frame, cand, changed=False, progress=False, danger=True)
            self.graph.record_test(prev_payload.frame_hash, cand.idx, success=False)
            return result

        state = obs_state(next_obs)
        next_completed = obs_levels_completed(next_obs)
        progress = next_completed > self.current_level_completed or state == GameState.WIN
        danger = state == GameState.GAME_OVER

        if progress:
            self.bandit.update(cand, changed=True, progress=True, danger=False)
            self.predictor.record(prev_payload.frame_hash, prev_payload.canonical_frame, cand, changed=True, progress=True, danger=False)
            result["changed"] = True
            result["progress"] = True
            return result

        if danger:
            self.bandit.update(cand, changed=False, progress=False, danger=True)
            self.predictor.record(prev_payload.frame_hash, prev_payload.canonical_frame, cand, changed=False, progress=False, danger=True)
            self.graph.record_test(prev_payload.frame_hash, cand.idx, success=False)
            result["danger"] = True
            return result

        next_payload = self._get_state_payload(next_obs)
        changed = next_payload.frame_hash != prev_payload.frame_hash
        self.bandit.update(cand, changed=changed, progress=False, danger=False)
        self.predictor.record(prev_payload.frame_hash, prev_payload.canonical_frame, cand, changed=changed, progress=False, danger=False)

        if changed:
            self.graph.record_test(
                prev_payload.frame_hash,
                cand.idx,
                success=True,
                target_node=next_payload.frame_hash,
                target_num_candidates=len(next_payload.candidates),
                target_group_sets=next_payload.group_sets,
            )
            result["changed"] = True
            result["next_payload"] = next_payload
        else:
            self.graph.record_test(prev_payload.frame_hash, cand.idx, success=False)
            result["next_payload"] = next_payload

        return result

    def run(self, env: Any, deadline: Optional[float] = None) -> dict:
        obs = env.observation_space
        if obs is None or obs_state(obs) == GameState.NOT_PLAYED:
            obs = env.reset()

        if obs is None:
            return {
                "game_id": self.game_id,
                "levels_completed": 0,
                "actions": 0,
                "state": "NOT_PLAYED",
            }

        self._new_level(obs)

        while True:
            if deadline is not None and time.time() >= deadline:
                break

            state = obs_state(obs)
            if state == GameState.WIN:
                break

            completed = obs_levels_completed(obs)
            if completed != self.current_level_completed:
                self._new_level(obs)
                state = obs_state(obs)

            if state == GameState.GAME_OVER:
                if self.should_retry(obs):
                    if deadline is not None and time.time() >= deadline:
                        break
                    obs = env.reset()
                    self.level_action_count += 1
                    self.local_total_actions += 1
                    self.local_total_resets += 1
                    self.level_reset_count += 1
                    if obs is None:
                        break
                    continue
                break

            if self.should_stop_level(obs):
                break

            payload = self._get_state_payload(obs)
            cand = self.choose_candidate(payload)
            if cand is None:
                break

            action = cand.action_enum()
            data = cand.action_data()

            next_obs = env.step(action, data=data)
            self.level_action_count += 1
            self.local_total_actions += 1
            transition = self.apply_transition(payload, cand, next_obs)

            if transition["progress"]:
                obs = next_obs
                if obs is None:
                    break
                if obs_state(obs) != GameState.WIN:
                    self._new_level(obs)
                continue

            if next_obs is None:
                break

            obs = next_obs

        final_state = obs_state(obs).name if obs is not None else "NOT_PLAYED"
        return {
            "game_id": self.game_id,
            "levels_completed": obs_levels_completed(obs) if obs is not None else 0,
            "actions": self.local_total_actions,
            "resets": self.local_total_resets,
            "state": final_state,
        }


arc = Arcade(
    operation_mode=OperationMode.OFFLINE,
    environments_dir=ENVIRONMENTS_DIR,
)

env_infos = arc.get_environments()
scorecard_id = arc.open_scorecard(tags=["agent", "hybrid", "graph-bandit"])

manual_results: List[dict] = []

logger.info(f"Operation mode: {arc.operation_mode}")
logger.info(f"Environment count: {len(env_infos)}")

start_time = time.time()
global_deadline = start_time + MAX_WALL_CLOCK_SECONDS

for order_idx, env_info in enumerate(env_infos):
    if time.time() >= global_deadline:
        logger.warning("global time budget reached, stopping remaining games")
        break
    game_id = env_info.game_id
    logger.info(f"[{order_idx + 1}/{len(env_infos)}] starting {game_id}")
    try:
        env = arc.make(
            game_id,
            scorecard_id=scorecard_id,
            save_recording=SAVE_RECORDINGS,
            include_frame_data=INCLUDE_FRAME_DATA,
        )
        if env is None:
            manual_results.append({
                "order": order_idx,
                "game_id": game_id,
                "score": 0.0,
                "levels_completed": 0,
                "actions": 0,
                "resets": 0,
                "completed": False,
                "state": "NOT_PLAYED",
                "level_scores": json.dumps([]),
                "level_actions": json.dumps([]),
                "level_baseline_actions": json.dumps(getattr(env_info, "baseline_actions", None)),
                "title": getattr(env_info, "title", None),
            })
            continue

        agent = HybridGameAgent(env_info, seed=SEED + order_idx * 997)
        local_result = agent.run(env, deadline=global_deadline)
        local_result["order"] = order_idx
        manual_results.append(local_result)
        logger.info(
            f"finished {game_id} | levels={local_result.get('levels_completed', 0)} | "
            f"actions={local_result.get('actions', 0)} | state={local_result.get('state', 'NA')}"
        )
        del agent
        del env
        gc.collect()
        if USE_TORCH_MODEL and torch is not None:
            torch.cuda.empty_cache()

    except Exception as exc:
        logger.exception(f"game {game_id} failed: {exc}")
        manual_results.append({
            "order": order_idx,
            "game_id": game_id,
            "score": 0.0,
            "levels_completed": 0,
            "actions": 0,
            "resets": 0,
            "completed": False,
            "state": "ERROR",
            "level_scores": json.dumps([]),
            "level_actions": json.dumps([]),
            "level_baseline_actions": json.dumps(getattr(env_info, "baseline_actions", None)),
            "title": getattr(env_info, "title", None),
        })
        continue

elapsed = time.time() - start_time
logger.info(f"Finished gameplay loop in {elapsed / 60:.2f} minutes")

final_scorecard = None
try:
    final_scorecard = arc.close_scorecard(scorecard_id)
except Exception as exc:
    logger.exception(f"close_scorecard failed: {exc}")

rows_by_game: Dict[str, dict] = {}
if final_scorecard is not None:
    try:
        Path("final_scorecard.json").write_text(final_scorecard.model_dump_json(indent=2, exclude_none=True))
    except Exception:
        pass

    for env_score_list in final_scorecard.environments:
        runs = list(env_score_list.runs)
        best_run = max(runs, key=lambda r: (r.levels_completed, r.score, -r.actions)) if runs else None
        rows_by_game[env_score_list.id] = {
            "game_id": env_score_list.id,
            "score": float(env_score_list.score),
            "levels_completed": int(env_score_list.levels_completed),
            "actions": int(env_score_list.actions),
            "resets": int(env_score_list.resets),
            "completed": bool(env_score_list.completed),
            "state": best_run.state.name if (best_run is not None and best_run.state is not None) else None,
            "level_scores": json.dumps(best_run.level_scores if best_run is not None and best_run.level_scores is not None else []),
            "level_actions": json.dumps(best_run.level_actions if best_run is not None and best_run.level_actions is not None else []),
            "level_baseline_actions": json.dumps(best_run.level_baseline_actions if best_run is not None and best_run.level_baseline_actions is not None else []),
        }

final_rows: List[dict] = []
manual_map = {row["game_id"]: row for row in manual_results}

for order_idx, env_info in enumerate(env_infos):
    game_id = env_info.game_id
    row = rows_by_game.get(game_id, {}).copy()
    if not row:
        local = manual_map.get(game_id, {})
        row = {
            "game_id": game_id,
            "score": float(local.get("score", 0.0)),
            "levels_completed": int(local.get("levels_completed", 0)),
            "actions": int(local.get("actions", 0)),
            "resets": int(local.get("resets", 0)),
            "completed": bool(local.get("completed", False)),
            "state": local.get("state", None),
            "level_scores": json.dumps(local.get("level_scores", [])) if not isinstance(local.get("level_scores", "[]"), str) else local.get("level_scores", "[]"),
            "level_actions": json.dumps(local.get("level_actions", [])) if not isinstance(local.get("level_actions", "[]"), str) else local.get("level_actions", "[]"),
            "level_baseline_actions": json.dumps(getattr(env_info, "baseline_actions", None)),
        }
    row["order"] = order_idx
    row["title"] = getattr(env_info, "title", None)
    row["baseline_actions"] = json.dumps(getattr(env_info, "baseline_actions", None))
    final_rows.append(row)

submission_columns = [
    "order",
    "game_id",
    "score",
    "levels_completed",
    "actions",
    "resets",
    "completed",
    "state",
    "level_scores",
    "level_actions",
    "level_baseline_actions",
    "title",
    "baseline_actions",
]
submission = pd.DataFrame(final_rows, columns=submission_columns)
if "order" in submission.columns:
    submission = submission.sort_values(["order", "game_id"], kind="stable").reset_index(drop=True)

parquet_written = False
for parquet_engine in [None, "pyarrow", "fastparquet"]:
    try:
        if parquet_engine is None:
            submission.to_parquet("submission.parquet", index=False)
        else:
            submission.to_parquet("submission.parquet", index=False, engine=parquet_engine)
        parquet_written = True
        break
    except Exception:
        continue

if not parquet_written:
    submission.to_csv("submission_fallback.csv", index=False)
    Path("submission.parquet").write_bytes(b"")
    print("parquet engine unavailable; wrote submission_fallback.csv and an empty submission.parquet placeholder")

print(f"submission shape: {submission.shape}")
if final_scorecard is not None:
    print(f"overall score: {float(final_scorecard.score):.6f}")
else:
    print("overall score: unavailable")


2026-03-30 12:44:24 | INFO | Created new scorecard: 2765ef79-7dee-4c06-8d4c-763141835aeb


2026-03-30 12:44:24,105 | INFO | Operation mode: OperationMode.OFFLINE
2026-03-30 12:44:24,105 | INFO | Environment count: 25
2026-03-30 12:44:24,106 | INFO | [1/25] starting sk48-41055498


2026-03-30 12:44:24 | INFO | Successfully loaded game class Sk48 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py


2026-03-30 12:44:31,931 | INFO | finished sk48-41055498 | levels=0 | actions=280 | state=NOT_FINISHED
2026-03-30 12:44:32,075 | INFO | [2/25] starting tn36-ab4f63cc


2026-03-30 12:44:32 | INFO | Successfully loaded game class Tn36 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/tn36.py


...

2026-03-30 12:47:45 | INFO | Successfully loaded game class Tr87 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tr87/cd924810/tr87.py


2026-03-30 12:47:51,845 | INFO | finished tr87-cd924810 | levels=0 | actions=296 | state=NOT_FINISHED
2026-03-30 12:47:51,968 | INFO | [25/25] starting g50t-5849a774


2026-03-30 12:47:52 | INFO | Successfully loaded game class G50t from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/g50t/5849a774/g50t.py


2026-03-30 12:47:58,344 | INFO | finished g50t-5849a774 | levels=0 | actions=284 | state=NOT_FINISHED
2026-03-30 12:47:58,476 | INFO | Finished gameplay loop in 3.57 minutes


2026-03-30 12:47:58 | INFO | Closed scorecard: 2765ef79-7dee-4c06-8d4c-763141835aeb
submission shape: (25, 13)
overall score: 0.046252


In [3]:
print(submission.head(25))
print("Average score of first 25 games:", float(submission.head(25)["score"].mean()))

    order        game_id     score  levels_completed  actions  resets  \
0       0  sk48-41055498  0.000000                 0      280       1   
1       1  tn36-ab4f63cc  0.000000                 0      260       4   
2       2  m0r0-dadda488  0.209581                 1      563       2   
3       3  bp35-0a0ad940  0.029136                 1      274       6   
4       4  cn04-65d47d14  0.000000                 0      260       1   
5       5  dc22-4c9bff3e  0.000000                 0      420       4   
6       6  tu93-2b534c15  0.000000                 0      220       4   
7       7  lp85-305b61c3  0.145882                 1      404       5   
8       8  ka59-9f096b4a  0.000000                 0      352       3   
9       9  wa30-ee6fef47  0.000000                 0      420       2   
10     10  vc33-9851e02b  0.313973                 2      541       7   
11     11  lf52-271a04aa  0.000000                 0      280       3   
12     12  r11l-aa269680  0.037387                 